In [13]:
# install dependencies 
print("Installing dependencies...")
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from matplotlib.ticker import FuncFormatter


import os 
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import logging
import sys
from pathlib import Path
from logging.handlers import RotatingFileHandler

def get_logger(
    name: str = __name__,
    log_file: str = "app.log",
    level: int = logging.DEBUG,
    max_bytes: int = 5 * 1024 * 1024,
    backup_count: int = 3
) -> logging.Logger:
    """
    Buat logger yang siap pakai dengan output ke console + file.

    Args:
        name        : nama logger, biasanya __name__ dari modul yang memanggil
        log_file    : path file log
        level       : minimum level yang direkam (default: DEBUG)
        max_bytes   : ukuran max per file log sebelum di-rotate (default: 5MB)
        backup_count: jumlah file backup yang disimpan

    Returns:
        logging.Logger
    """

    # Make sure log file exists
    Path(log_file).parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(name)

    # Avoid duplicate handler, if handler exists, use the existing handler
    if logger.handlers:
        return logger

    logger.setLevel(level)

    # logging format
    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(filename)s:%(lineno)d | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    # handler for console stdout
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)   # console: INFO ke atas saja
    console_handler.setFormatter(fmt)

    # handler for log file
    file_handler = RotatingFileHandler(
        log_file,
        maxBytes=max_bytes,
        backupCount=backup_count,
        encoding="utf-8"
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(fmt)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

    return logger

# get root path of the project
PATH = Path(os.path.dirname(os.getcwd()))

print("All dependencies installed successfully!")
print(f"Current project directory: {PATH}")

Installing dependencies...
All dependencies installed successfully!
Current project directory: /Users/nadif/projects/hdi-recruitment


In [15]:
# building preprocessing data pipeline

def preprocessing_daily_ops(input_path: str, output_path: str = "data/processed/hdi_daily_ops_clean.csv") -> pd.DataFrame:
  logger = get_logger(__name__, log_file="logs/preprocessing.log")
  
  """
  Preprocessing daily operational data. Returns a clean, validated pandas dataframe.
  """
  try:
    logger.info(f"Preprocessing raw csv data from {input_path}...")
    
    # read the data
    df = pd.read_csv(input_path)
    logger.info(f"Dataset succesfully read. {df.shape[0]} rows and {df.shape[1]} columns of daily operational data has been successfully loaded!")

    # change date from string to datetime type
    logger.info(f"Change 'date' columns data type to datetime")
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
    invalid_dates_counts = df.isna().sum()
    if invalid_dates_counts.sum():
      logger.warning(f"{invalid_dates_counts} rows can't be parsed! Proceed to drop the rows...")
      df.dropna(subset=["date"])
    logger.info("'date' columns data type has been changed to datetime!")

    # check date continuity
    logger.info("Checking date continuity...")
    min_date = df["date"].min()
    max_date = df["date"].max()
    date_full_range = pd.date_range(min_date, max_date, freq="D")
    missing_dates = date_full_range.difference(df["date"])
    if len(missing_dates) > 0:
      logger.warning(f"{len(missing_dates)} missing dates found: {missing_dates.to_list()}...")
    else:
      logger.info("There is no gap from date data!")

    # handling missing values 
    logger.info("Handling missing values...")
    missing_value_counts = df.isna().sum()
    if missing_value_counts.sum():
      logger.warning(f"Missing value found: {missing_value_counts[missing_value_counts > 0]}")
      if "day_of_week" in missing_value_counts[missing_value_counts > 0].to_list():
        logger.warning(f"Proceed to fill day_of_week from date data...")
        df["date"] = df["date"].fillna(df["date"].dt.day_name())
      else:
        logger.warning(f"To prevent data leakage in training, missing values \
         beside 'day_of_week' columns will be left unfilled!")
    else:
      logger.info("There is no missing value in the data!")
    
    # validate the day 
    logger.info("Validate the day, should match the 'date' column...")
    expected_day = df["date"].dt.day_name()
    mismatch = (df["day_of_week"] != expected_day).sum()
    if mismatch > 0:
      logger.warning(f"{mismatch} day_of_week data does not match the date! Proceed to fix..")
      df["day_of_week"] = expected_day
    else:
      logger.info("All days match the date!")
    
    # remove duplicate rows
    logger.info("Check for the duplicate data...")
    len_before = len(df)
    df = df.drop_duplicates(subset=["date"]).reset_index(drop=True)
    if len_before > len(df):
      logger.warning(f"{len_before - len(df)} duplicate(s) data found! Proceed to remove...")
    else:
      logger.info("There is no duplicate data in the dataset.")
    
    # flag the outlier in 'new_enterpriser_count', 'new_bee_count', and 'sales_ep_thousand_idr' columns
    for col in ['new_enterpriser_count', 'new_bee_count', 'sales_ep_thousand_idr']:
      q1, q3 = df[col].quantile([.25,.75])
      iqr = q3 - q1
      lower, upper = q1 - 3 * iqr, q3 + 3 * iqr 
      outliers = df[(df[col] > upper) | (df[col] < lower)]
      if len(outliers) > 0:
        logger.info(f"There are {len(outliers)} outlier(s) in {col} data! Proceed to flag it.")
        df[f"{col}_outlier"] = ((df[col] > upper) | (df[col] < lower)).astype(int)
      else:
        logger.info(f"There is no outliers in {col} column!")
    
    # save the dataset
    df.to_csv(output_path, index=False)
    logger.info(f"Dataset cleaning done! Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns and saved to {output_path}")
    return df
  except FileNotFoundError:
    logger.error(f"Can't find {input_path}!")
  except Exception as e:
    logger.critical(f"Unexpected error when loading data: {e}", exc_info=True)

In [16]:
DATA_DIR = PATH / "data" 
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
df = preprocessing_daily_ops(RAW_DIR / "hdi_daily_ops.csv", PROCESSED_DIR / "hdi_daily_ops_cleaned.csv")

2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:10 | Preprocessing raw csv data from /Users/nadif/projects/hdi-recruitment/data/raw/hdi_daily_ops.csv...
2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:14 | Dataset succesfully read. 1631 rows and 9 columns of daily operational data has been successfully loaded!
2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:17 | Change 'date' columns data type to datetime
2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:23 | 'date' columns data type has been changed to datetime!
2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:26 | Checking date continuity...
2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:34 | There is no gap from date data!
2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:37 | Handling missing values...
2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:48 | There is no missing value in the data!
2026-06-28 03:04:22 | INFO     | __main__ | 2098734338.py:51 | Va